# Model Results - Pareto Front Visualizations

All charts load from `all_model_results.csv` (195 rows: 105 Baseline GA / 30 Coevolution GA / 30 NSGA-II / 30 Greedy Builder).

**Compound-rule methods** (Set 2): Coevolution GA, NSGA-II, Greedy Builder — all produce OR-of-AND rule sets.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings("ignore")

OUTPUT_DIR = r"C:/Users/schyu/Downloads/OneDrive_1_3-10-2026/outputs"

df_all = pd.read_csv(f"{OUTPUT_DIR}/all_model_results.csv")

df_baseline = df_all[df_all["model"] == "Baseline GA"].copy().reset_index(drop=True)
df_coevo    = df_all[df_all["model"] == "Coevolution GA"].copy().reset_index(drop=True)
df_nsga2    = df_all[df_all["model"] == "NSGA-II"].copy().reset_index(drop=True)
df_greedy   = df_all[df_all["model"] == "Greedy Builder"].copy().reset_index(drop=True)

print(f"Loaded {len(df_all)} total rows")
print(f"  Baseline GA:    {len(df_baseline)} variants")
print(f"  Coevolution GA: {len(df_coevo)} variants")
print(f"  NSGA-II:        {len(df_nsga2)} variants")
print(f"  Greedy Builder: {len(df_greedy)} variants")

In [ ]:
# ── Palette ──────────────────────────────────────────────────────────────
MODEL_COLORS = {
    "Baseline GA":    "#D32F2F",
    "Coevolution GA": "#FF6F00",
    "NSGA-II":        "#1565C0",
    "Greedy Builder": "#2E7D32",
}
CX_PALETTE  = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd","#8c564b","#e377c2"]
MUT_PALETTE = ["#17becf","#bcbd22","#7f7f7f"]
SEL_PALETTE = ["#1f77b4","#ff7f0e","#2ca02c","#d62728","#9467bd"]

# ── Pareto helpers ────────────────────────────────────────────────────────
def pareto_front_mask(prec, rec):
    pts = np.column_stack([np.asarray(prec, float), np.asarray(rec, float)])
    n = len(pts)
    mask = np.ones(n, dtype=bool)
    for i in range(n):
        if not mask[i]:
            continue
        for j in range(n):
            if i == j or not mask[j]:
                continue
            if (pts[j,0] >= pts[i,0] and pts[j,1] >= pts[i,1] and
                    (pts[j,0] > pts[i,0] or pts[j,1] > pts[i,1])):
                mask[i] = False
                break
    return mask

def draw_pareto_step(ax, prec, rec, color, lw=1.8, alpha=0.75):
    pts = sorted(zip(rec, prec))
    if len(pts) < 2:
        return
    ax.plot([p[0] for p in pts], [p[1] for p in pts],
            color=color, lw=lw, alpha=alpha, zorder=2)

def tight_limits(ax, rec_vals, prec_vals, pad=0.06):
    r_min, r_max = float(np.nanmin(rec_vals)), float(np.nanmax(rec_vals))
    p_min, p_max = float(np.nanmin(prec_vals)), float(np.nanmax(prec_vals))
    r_rng = max(r_max - r_min, 0.04)
    p_rng = max(p_max - p_min, 0.04)
    ax.set_xlim(r_min - pad*r_rng, r_max + pad*r_rng)
    ax.set_ylim(p_min - pad*p_rng, p_max + pad*p_rng)

def plot_group(ax, df, prec_col, rec_col, f1_col, color, label,
               dot_size=35, dot_alpha=0.55, star_size=380, draw_front=True):
    df_v = df.dropna(subset=[prec_col, rec_col, f1_col])
    if df_v.empty:
        return
    if draw_front and len(df_v) > 1:
        pf = pareto_front_mask(df_v[prec_col].values, df_v[rec_col].values)
        if pf.sum() > 1:
            draw_pareto_step(ax, df_v[prec_col].values[pf],
                             df_v[rec_col].values[pf], color=color)
    best_idx = df_v[f1_col].idxmax()
    normal   = df_v[df_v.index != best_idx]
    ax.scatter(normal[rec_col], normal[prec_col],
               c=color, s=dot_size, alpha=dot_alpha,
               edgecolors="white", linewidths=0.3, zorder=3)
    ax.scatter(df_v.loc[best_idx, rec_col], df_v.loc[best_idx, prec_col],
               c=color, s=star_size, marker="*",
               edgecolors="black", linewidths=0.5, zorder=6, label=label)
    # Mean square
    mean_rec  = df_v[rec_col].mean()
    mean_prec = df_v[prec_col].mean()
    ax.scatter(mean_rec, mean_prec,
               c=color, s=120, marker="s",
               edgecolors="black", linewidths=0.8, zorder=7, alpha=0.9)

def style_ax(ax, title, xlabel="Recall", ylabel="Precision", legend_loc="upper right"):
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(title, fontsize=10, fontweight="bold")
    ax.grid(True, alpha=0.25, linestyle="--")
    ax.legend(fontsize=7, loc=legend_loc, framealpha=0.85, markerscale=0.7)

print("Helpers ready.")

In [ ]:
# Set 1 - Figure 1: Baseline GA grid coloured by Crossover operator
cx_ops  = sorted(df_baseline["crossover"].dropna().unique())
palette = dict(zip(cx_ops, CX_PALETTE[:len(cx_ops)]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for split, (label, prec_col, rec_col, f1_col) in enumerate([
    ("Val",  "val_precision",  "val_recall",  "val_f1"),
    ("Test", "test_precision", "test_recall", "test_f1"),
]):
    ax = axes[split]
    all_r, all_p = [], []
    for cx in cx_ops:
        sub = df_baseline[df_baseline["crossover"] == cx]
        plot_group(ax, sub, prec_col, rec_col, f1_col, palette[cx], cx)
        all_r.extend(sub[rec_col].dropna()); all_p.extend(sub[prec_col].dropna())
    tight_limits(ax, all_r, all_p)
    style_ax(ax, f"Baseline GA Grid - {label} Set\n(colour = crossover operator)")

fig.suptitle("Set 1 - Fig 1: Grid Search Pareto Front by Crossover  (* = best F1 per operator)",
             fontsize=12, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/set1_fig1_pareto_crossover.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved set1_fig1_pareto_crossover.png")

In [ ]:
# Set 1 - Figure 2: Baseline GA grid coloured by Mutation operator
mut_ops = sorted(df_baseline["mutation"].dropna().unique())
palette = dict(zip(mut_ops, MUT_PALETTE[:len(mut_ops)]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for split, (label, prec_col, rec_col, f1_col) in enumerate([
    ("Val",  "val_precision",  "val_recall",  "val_f1"),
    ("Test", "test_precision", "test_recall", "test_f1"),
]):
    ax = axes[split]
    all_r, all_p = [], []
    for mut in mut_ops:
        sub = df_baseline[df_baseline["mutation"] == mut]
        plot_group(ax, sub, prec_col, rec_col, f1_col, palette[mut], mut)
        all_r.extend(sub[rec_col].dropna()); all_p.extend(sub[prec_col].dropna())
    tight_limits(ax, all_r, all_p)
    style_ax(ax, f"Baseline GA Grid - {label} Set\n(colour = mutation operator)")

fig.suptitle("Set 1 - Fig 2: Grid Search Pareto Front by Mutation  (* = best F1 per operator)",
             fontsize=12, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/set1_fig2_pareto_mutation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved set1_fig2_pareto_mutation.png")

In [ ]:
# Set 1 - Figure 3: Baseline GA grid coloured by Selection operator
sel_ops = sorted(df_baseline["selection"].dropna().unique())
palette = dict(zip(sel_ops, SEL_PALETTE[:len(sel_ops)]))

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for split, (label, prec_col, rec_col, f1_col) in enumerate([
    ("Val",  "val_precision",  "val_recall",  "val_f1"),
    ("Test", "test_precision", "test_recall", "test_f1"),
]):
    ax = axes[split]
    all_r, all_p = [], []
    for sel in sel_ops:
        sub = df_baseline[df_baseline["selection"] == sel]
        plot_group(ax, sub, prec_col, rec_col, f1_col, palette[sel], sel)
        all_r.extend(sub[rec_col].dropna()); all_p.extend(sub[prec_col].dropna())
    tight_limits(ax, all_r, all_p)
    style_ax(ax, f"Baseline GA Grid - {label} Set\n(colour = selection method)")

fig.suptitle("Set 1 - Fig 3: Grid Search Pareto Front by Selection  (* = best F1 per method)",
             fontsize=12, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/set1_fig3_pareto_selection.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved set1_fig3_pareto_selection.png")

In [ ]:
# Set 2 - Figure 1: Compound-rule methods — Coevolution GA vs NSGA-II vs Greedy Builder
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

compound_families = [
    ("Coevolution GA", df_coevo),
    ("NSGA-II",        df_nsga2),
    ("Greedy Builder", df_greedy),
]

for split, (label, prec_col, rec_col, f1_col) in enumerate([
    ("Val",  "val_precision",  "val_recall",  "val_f1"),
    ("Test", "test_precision", "test_recall", "test_f1"),
]):
    ax = axes[split]
    all_r, all_p = [], []
    for key, df_fam in compound_families:
        color = MODEL_COLORS[key]
        n = len(df_fam.dropna(subset=[prec_col]))
        plot_group(ax, df_fam, prec_col, rec_col, f1_col,
                   color, f"{key} (n={n})")
        all_r.extend(df_fam[rec_col].dropna())
        all_p.extend(df_fam[prec_col].dropna())
    tight_limits(ax, all_r, all_p)
    style_ax(ax, f"Compound-Rule Methods - {label} Set\n(* = best F1 per family)",
             legend_loc="upper left")

fig.suptitle("Set 2 - Fig 1: Coevolution GA (30) vs NSGA-II (30) vs Greedy Builder (30)  (* = best F1 per family)",
             fontsize=12, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/set2_fig1_coevo_vs_nsga2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved set2_fig1_coevo_vs_nsga2.png")

In [ ]:
# Set 2 - Figure 2: All four model families (105 + 30 + 30 + 30 = 195 points)
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

families = [
    ("Baseline GA",    df_baseline),
    ("Coevolution GA", df_coevo),
    ("NSGA-II",        df_nsga2),
    ("Greedy Builder", df_greedy),
]

for split, (label, prec_col, rec_col, f1_col) in enumerate([
    ("Val",  "val_precision",  "val_recall",  "val_f1"),
    ("Test", "test_precision", "test_recall", "test_f1"),
]):
    ax = axes[split]
    all_r, all_p = [], []
    for key, df_fam in families:
        color = MODEL_COLORS[key]
        n = len(df_fam.dropna(subset=[prec_col]))
        plot_group(ax, df_fam, prec_col, rec_col, f1_col,
                   color, f"{key} (n={n})", dot_size=25, dot_alpha=0.45)
        all_r.extend(df_fam[rec_col].dropna())
        all_p.extend(df_fam[prec_col].dropna())
    tight_limits(ax, all_r, all_p)
    style_ax(ax, f"All Model Families - {label} Set\n(* = best F1 per family)",
             legend_loc="upper left")

fig.suptitle(
    "Set 2 - Fig 2: All Models — Baseline GA (105) / Coevolution GA (30) / NSGA-II (30) / Greedy Builder (30)  (* = best F1)",
    fontsize=12, fontweight="bold", y=1.01)
fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/set2_fig2_all_models.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved set2_fig2_all_models.png")

In [ ]:
# ── Standalone test-set figures ───────────────────────────────────────────────
# Each entry: (filename_stem, title, families, dot_size, dot_alpha, legend_loc, group_by)
# group_by = None means use families list directly;
# group_by = column name means split df_baseline by that column

def save_test_figure(stem, suptitle, families, dot_size=35, dot_alpha=0.55, legend_loc="upper left"):
    fig, ax = plt.subplots(figsize=(7, 5.5))
    all_r, all_p = [], []
    for key, df_fam, color in families:
        n = len(df_fam.dropna(subset=["test_precision"]))
        plot_group(ax, df_fam, "test_precision", "test_recall", "test_f1",
                   color, f"{key} (n={n})", dot_size=dot_size, dot_alpha=dot_alpha)
        all_r.extend(df_fam["test_recall"].dropna())
        all_p.extend(df_fam["test_precision"].dropna())
    tight_limits(ax, all_r, all_p)
    style_ax(ax, "Test Set", legend_loc=legend_loc)
    fig.suptitle(suptitle, fontsize=11, fontweight="bold", y=1.01)
    fig.tight_layout()
    out = f"{OUTPUT_DIR}/{stem}_test.png"
    fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved {stem}_test.png")

# Set 1 Fig 1 — by crossover
cx_ops = sorted(df_baseline["crossover"].dropna().unique())
cx_pal = dict(zip(cx_ops, CX_PALETTE[:len(cx_ops)]))
save_test_figure(
    "set1_fig1_pareto_crossover",
    "Set 1 - Fig 1 (Test): Baseline GA Grid by Crossover",
    [(cx, df_baseline[df_baseline["crossover"] == cx], cx_pal[cx]) for cx in cx_ops],
    legend_loc="upper right",
)

# Set 1 Fig 2 — by mutation
mut_ops = sorted(df_baseline["mutation"].dropna().unique())
mut_pal = dict(zip(mut_ops, MUT_PALETTE[:len(mut_ops)]))
save_test_figure(
    "set1_fig2_pareto_mutation",
    "Set 1 - Fig 2 (Test): Baseline GA Grid by Mutation",
    [(m, df_baseline[df_baseline["mutation"] == m], mut_pal[m]) for m in mut_ops],
    legend_loc="upper right",
)

# Set 1 Fig 3 — by selection
sel_ops = sorted(df_baseline["selection"].dropna().unique())
sel_pal = dict(zip(sel_ops, SEL_PALETTE[:len(sel_ops)]))
save_test_figure(
    "set1_fig3_pareto_selection",
    "Set 1 - Fig 3 (Test): Baseline GA Grid by Selection",
    [(s, df_baseline[df_baseline["selection"] == s], sel_pal[s]) for s in sel_ops],
    legend_loc="upper right",
)

# Set 2 Fig 1 — compound-rule methods
save_test_figure(
    "set2_fig1_coevo_vs_nsga2",
    "Set 2 - Fig 1 (Test): Coevolution GA vs NSGA-II vs Greedy Builder",
    [(k, d, MODEL_COLORS[k]) for k, d in [
        ("Coevolution GA", df_coevo),
        ("NSGA-II",        df_nsga2),
        ("Greedy Builder", df_greedy),
    ]],
)

# Set 2 Fig 2 — all families
save_test_figure(
    "set2_fig2_all_models",
    "Set 2 - Fig 2 (Test): All Model Families",
    [(k, d, MODEL_COLORS[k]) for k, d in [
        ("Baseline GA",    df_baseline),
        ("Coevolution GA", df_coevo),
        ("NSGA-II",        df_nsga2),
        ("Greedy Builder", df_greedy),
    ]],
    dot_size=25, dot_alpha=0.45,
)

In [ ]:
# ── Presentation figure: All model families — Test Set ────────────────────────
import matplotlib.lines as mlines

fig, ax = plt.subplots(figsize=(10, 6))   # 5:3 ratio

for key, df_fam in [
    ("Baseline GA",    df_baseline),
    ("Coevolution GA", df_coevo),
    ("NSGA-II",        df_nsga2),
    ("Greedy Builder", df_greedy),
]:
    color = MODEL_COLORS[key]
    df_v  = df_fam.dropna(subset=["test_precision", "test_recall", "test_f1"])
    if df_v.empty:
        continue
    if len(df_v) > 1:
        pf = pareto_front_mask(df_v["test_precision"].values, df_v["test_recall"].values)
        if pf.sum() > 1:
            draw_pareto_step(ax, df_v["test_precision"].values[pf],
                             df_v["test_recall"].values[pf], color=color)
    best_idx = df_v["test_f1"].idxmax()
    normal   = df_v[df_v.index != best_idx]
    ax.scatter(normal["test_recall"], normal["test_precision"],
               c=color, s=28, alpha=0.45, edgecolors="white", linewidths=0.3, zorder=3)
    ax.scatter(df_v.loc[best_idx, "test_recall"], df_v.loc[best_idx, "test_precision"],
               c=color, s=380, marker="*",
               edgecolors="black", linewidths=0.5, zorder=6, label=key)
    ax.scatter(df_v["test_recall"].mean(), df_v["test_precision"].mean(),
               c=color, s=120, marker="s",
               edgecolors="black", linewidths=0.8, zorder=7, alpha=0.9)

all_r = pd.concat([df_baseline, df_coevo, df_nsga2, df_greedy])["test_recall"].dropna()
all_p = pd.concat([df_baseline, df_coevo, df_nsga2, df_greedy])["test_precision"].dropna()
tight_limits(ax, all_r, all_p)

ax.set_xlabel("Recall", fontsize=12)
ax.set_ylabel("Precision", fontsize=12)
ax.set_title(
    "Precision–Recall Pareto Front Comparison\nAcross Evolutionary Rule-Learning Methods (Test Set)",
    fontsize=12, fontweight="bold"
)

# Build legend: model entries + marker-meaning entries
star_handle   = mlines.Line2D([], [], color="grey", marker="*", linestyle="None",
                               markersize=11, markeredgecolor="black",
                               markeredgewidth=0.5, label="Best F1")
square_handle = mlines.Line2D([], [], color="grey", marker="s", linestyle="None",
                               markersize=8, markeredgecolor="black",
                               markeredgewidth=0.8, label="Group mean")

handles, labels = ax.get_legend_handles_labels()
handles += [star_handle, square_handle]

ax.legend(handles=handles, fontsize=10, loc="upper left",
          framealpha=0.9, markerscale=0.75)
ax.grid(True, alpha=0.25, linestyle="--")

fig.tight_layout()
fig.savefig(f"{OUTPUT_DIR}/presentation_all_models_test.png", dpi=200, bbox_inches="tight")
plt.show()
print("Saved presentation_all_models_test.png")

In [ ]:
# ── Presentation figures: Baseline GA operator comparisons — Test Set ─────────
import matplotlib.lines as mlines

def save_presentation_operator_fig(stem, title, groups, palette):
    fig, ax = plt.subplots(figsize=(10, 6))  # 5:3
    all_r, all_p = [], []
    for key, df_fam in groups:
        color = palette[key]
        df_v  = df_fam.dropna(subset=["test_precision", "test_recall", "test_f1"])
        if df_v.empty:
            continue
        if len(df_v) > 1:
            pf = pareto_front_mask(df_v["test_precision"].values, df_v["test_recall"].values)
            if pf.sum() > 1:
                draw_pareto_step(ax, df_v["test_precision"].values[pf],
                                 df_v["test_recall"].values[pf], color=color)
        best_idx = df_v["test_f1"].idxmax()
        normal   = df_v[df_v.index != best_idx]
        ax.scatter(normal["test_recall"], normal["test_precision"],
                   c=color, s=28, alpha=0.45, edgecolors="white", linewidths=0.3, zorder=3)
        ax.scatter(df_v.loc[best_idx, "test_recall"], df_v.loc[best_idx, "test_precision"],
                   c=color, s=380, marker="*",
                   edgecolors="black", linewidths=0.5, zorder=6, label=key)
        ax.scatter(df_v["test_recall"].mean(), df_v["test_precision"].mean(),
                   c=color, s=120, marker="s",
                   edgecolors="black", linewidths=0.8, zorder=7, alpha=0.9)
        all_r.extend(df_v["test_recall"]); all_p.extend(df_v["test_precision"])

    tight_limits(ax, all_r, all_p)
    ax.set_xlabel("Recall", fontsize=12)
    ax.set_ylabel("Precision", fontsize=12)
    ax.set_title(title, fontsize=12, fontweight="bold")

    star_handle   = mlines.Line2D([], [], color="grey", marker="*", linestyle="None",
                                   markersize=11, markeredgecolor="black",
                                   markeredgewidth=0.5, label="Best F1")
    square_handle = mlines.Line2D([], [], color="grey", marker="s", linestyle="None",
                                   markersize=8,  markeredgecolor="black",
                                   markeredgewidth=0.8, label="Group mean")
    handles, _ = ax.get_legend_handles_labels()
    ax.legend(handles=handles + [star_handle, square_handle],
              fontsize=10, loc="upper left", framealpha=0.9, markerscale=0.75)
    ax.grid(True, alpha=0.25, linestyle="--")
    fig.tight_layout()
    fig.savefig(f"{OUTPUT_DIR}/{stem}.png", dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved {stem}.png")

# -- Crossover --
cx_ops = sorted(df_baseline["crossover"].dropna().unique())
cx_pal = dict(zip(cx_ops, CX_PALETTE[:len(cx_ops)]))
save_presentation_operator_fig(
    "presentation_crossover_test",
    "Baseline GA: Precision–Recall Trade-off by Crossover Operator (Test Set)",
    [(cx, df_baseline[df_baseline["crossover"] == cx]) for cx in cx_ops],
    cx_pal,
)

# -- Mutation --
mut_ops = sorted(df_baseline["mutation"].dropna().unique())
mut_pal = dict(zip(mut_ops, MUT_PALETTE[:len(mut_ops)]))
save_presentation_operator_fig(
    "presentation_mutation_test",
    "Baseline GA: Precision–Recall Trade-off by Mutation Operator (Test Set)",
    [(m, df_baseline[df_baseline["mutation"] == m]) for m in mut_ops],
    mut_pal,
)

# -- Selection --
sel_ops = sorted(df_baseline["selection"].dropna().unique())
sel_pal = dict(zip(sel_ops, SEL_PALETTE[:len(sel_ops)]))
save_presentation_operator_fig(
    "presentation_selection_test",
    "Baseline GA: Precision–Recall Trade-off by Selection Method (Test Set)",
    [(s, df_baseline[df_baseline["selection"] == s]) for s in sel_ops],
    sel_pal,
)

In [ ]:
import os
print(f"Outputs in: {OUTPUT_DIR}")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    kb = os.path.getsize(fpath) / 1024
    print(f"  {fname:<50} {kb:.1f} KB")